# Google Trends Search Interest vs. Stock Price

The companion `wikipediaAttentionResearch` notebook used Wikipedia pageviews as an attention proxy.
Google Trends is the more direct version of the same idea — arguably closer to purchase intent,
since people literally search "TSLA stock" or "should I buy GameStop" rather than reading an
encyclopedia article. This notebook repeats that analysis with Google Trends as the data source, on
the **same four tickers** as the Wikipedia notebook, so the two independent attention proxies can be
checked against each other directly — if they agree, that's real signal; if they diverge, that tells
us something about which proxy (or neither) is trustworthy.

**Data source**: Google Trends, via the unofficial `pytrends` library. Two real constraints shape
this notebook's structure, both different from every prior notebook in this repo:

1. **No official API**: unlike TSA/Wikimedia/BTS (all designed for reuse), Google has no public
   Trends API. `pytrends` works by replaying the same requests the public trends.google.com website
   makes — it can break without warning if Google changes something, and it's more rate-limit-prone
   than the other sources. We add retry/backoff and keep request volume low.
2. **Granularity depends on window length, and downsamples further than you'd expect**: Google
   returns **daily** resolution for windows of 270 days or less. Longer ranges are downsampled — we
   verified a 4-year window still comes back **weekly**, but reaching back to 2019 (~7.5 years, needed
   to cover the GME squeeze) downsamples all the way to **monthly** (confirmed: 92 data points spaced
   exactly one month apart, not ~390 weekly points). We use both: a **monthly** long history
   (2019-present, for regime-level trends and the GME squeeze) and a short **daily** window (~9
   months, for lead-lag/event-study granularity) — there's a resolution gap between them (no clean
   "weekly, multi-year" tier available without stitching together many overlapping requests, which
   we don't do here).

**Tickers** (identical to the Wikipedia notebook, using Google's *disambiguated entity* codes, not
raw search terms — "Apple" the fruit vs. Apple Inc. matters, and "Tesla" collides with the physical
unit and the historical inventor): **TSLA**, **GME**, **COIN**, **AAPL**.

**Hypotheses:**
1. Search-interest growth correlates with returns, per ticker, at both monthly and daily granularity.
2. Lead-lag: does search interest lead price (people research before buying) or does price lead
   search interest (people search *because* something moved)?
3. **Cross-validation**: does Google Trends agree with Wikipedia pageviews for the same tickers over
   the same windows? Agreement between two independently-collected attention proxies is much stronger
   evidence than either one alone.
4. GME's Jan 2021 squeeze should appear as an extreme, unmistakable spike in monthly search interest —
   a sanity check that this data pipeline captures something real.

> ⚠️ Research only, not investment advice.


In [ ]:
# If needed, install dependencies (uncomment to run):
# %pip install pytrends yfinance pandas numpy matplotlib seaborn scipy statsmodels requests lxml

import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from pytrends.request import TrendReq

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="darkgrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)

pytrends = TrendReq(hl="en-US", tz=360)


## 1. Entity Disambiguation

Google Trends' `suggestions()` endpoint maps a plain-text query to disambiguated entities (each with
a stable `mid` code) — this is what lets us separate "Apple the company" from "apple the fruit," or
"Tesla the car company" from "tesla the unit of magnetic flux density." We look up each ticker's
company entity here for transparency, then use hardcoded, manually-verified `mid` codes below (the
suggestion *ranking* isn't guaranteed stable across Google Trends updates, so pinning verified codes
is more reproducible than re-resolving them fresh on every run).


In [ ]:
COMPANIES = {"TSLA": "Tesla, Inc.", "GME": "GameStop", "COIN": "Coinbase", "AAPL": "Apple Inc."}


def with_retry(fn, retries=5, backoff=20):
    for attempt in range(retries):
        try:
            return fn()
        except Exception as e:
            if attempt == retries - 1:
                raise
            wait = backoff * (attempt + 1)
            print(f"    retry {attempt + 1}/{retries} after error ({e}); waiting {wait}s")
            time.sleep(wait)


for ticker, query in COMPANIES.items():
    suggestions = with_retry(lambda q=query: pytrends.suggestions(q))
    print(f"{ticker} ({query}):")
    for s in suggestions[:3]:
        print(f"    {s['mid']}  {s['title']!r}  [{s['type']}]")
    time.sleep(1)


In [ ]:
# Manually verified against the suggestions() output above — the specific company entity for each,
# not a generic "Topic" grouping or an unrelated same-name entity.
ENTITY_MID = {
    "TSLA": "/m/0dr90d",   # Tesla, Inc. (Automotive company)
    "GME": "/m/03xlfx",    # GameStop (Electronics retail company)
    "COIN": "/m/0wr3qjq",  # Coinbase (Company)
    "AAPL": "/m/0k8z",     # Apple (Technology company) — disambiguated from the fruit
}


## 2. Data Collection

We fetch each ticker **separately** (one entity per request) rather than combining all four into one
call — Google Trends normalizes a combined request's results *relative to each other*, and Tesla's
search volume is large enough to compress the other three down near zero resolution (verified: a
combined test request left Apple Inc. maxing out at 1 out of 100). Separate requests keep each
ticker's own 0-100 scale meaningful.

A simple retry-with-backoff wrapper handles `pytrends`' occasional rate-limit hiccups.


In [ ]:
def fetch_trends(mid, timeframe):
    def _fetch():
        pytrends.build_payload([mid], timeframe=timeframe)
        df = pytrends.interest_over_time()
        if df.empty:
            return df
        return df.rename(columns={mid: "interest"})[["interest"]]
    return with_retry(_fetch)


TODAY = pd.Timestamp.today().normalize()
MONTHLY_START = "2019-01-01"
DAILY_WINDOW_DAYS = 260  # safely under Google's ~270-day daily-resolution cutoff
DAILY_START = (TODAY - pd.Timedelta(days=DAILY_WINDOW_DAYS)).strftime("%Y-%m-%d")

trends_monthly, trends_daily = {}, {}
for ticker, mid in ENTITY_MID.items():
    print(f"Fetching {ticker} monthly history...")
    trends_monthly[ticker] = fetch_trends(mid, f"{MONTHLY_START} {TODAY.strftime('%Y-%m-%d')}")
    time.sleep(1.5)
    print(f"Fetching {ticker} daily recent window...")
    trends_daily[ticker] = fetch_trends(mid, f"{DAILY_START} {TODAY.strftime('%Y-%m-%d')}")
    time.sleep(1.5)

print(f"\nMonthly points per ticker: {[len(v) for v in trends_monthly.values()]}")
print(f"Daily points per ticker: {[len(v) for v in trends_daily.values()]}")


In [ ]:
import yfinance as yf

stock_raw = yf.download(
    list(COMPANIES.keys()), start=MONTHLY_START, auto_adjust=True, progress=False, group_by="ticker"
)
stock = {}
for t in COMPANIES:
    d = stock_raw[t][["Close", "Volume"]].rename(columns={"Close": f"{t}_close", "Volume": f"{t}_vol"})
    stock[t] = d

stock_prices = pd.concat(stock.values(), axis=1)
stock_prices.index = pd.to_datetime(stock_prices.index)
stock_prices = stock_prices.sort_index()
stock_prices.tail()


## 3. Calendar Alignment

Same structural issue as every prior notebook: Google Trends is measured every day of the week (and,
for the long-history request, aggregated into monthly buckets labeled by month-start); stocks trade
~5 days/week. For the daily window we roll non-trading-day search interest forward into the next
trading day, as before. For the monthly window we resample stock closes to the last trading day of
each calendar month, labeled the same month-start way, so each row pairs "this month's search
interest" with "this month's stock return."


In [ ]:
def roll_forward_to_trading_days(daily_series, trading_day_index):
    full_idx = daily_series.index.union(trading_day_index)
    s = daily_series.reindex(full_idx).sort_index()
    bucket = pd.Series(pd.NaT, index=s.index, dtype="object")
    is_trading_day = bucket.index.isin(trading_day_index)
    bucket.loc[is_trading_day] = bucket.index[is_trading_day]
    bucket = bucket.bfill()
    return s.groupby(bucket).sum(min_count=1)


trading_days = pd.Index(stock_prices.dropna(how="all").index)

daily_aligned = {}
for t in COMPANIES:
    rolled = roll_forward_to_trading_days(trends_daily[t]["interest"], trading_days)
    daily_aligned[t] = rolled.reindex(trading_days)

df_daily = stock_prices.copy()
for t in COMPANIES:
    df_daily[f"{t}_trends"] = daily_aligned[t]
df_daily = df_daily.dropna(subset=[f"{t}_close" for t in COMPANIES], how="all")
df_daily.tail()


In [ ]:
monthly_close = stock_prices[[f"{t}_close" for t in COMPANIES]].resample("MS").last()
monthly_rets = np.log(monthly_close).diff()
monthly_rets.columns = [c.replace("_close", "_ret") for c in monthly_rets.columns]

df_monthly = monthly_rets.copy()
for t in COMPANIES:
    df_monthly[f"{t}_trends"] = trends_monthly[t]["interest"].reindex(df_monthly.index)
df_monthly = df_monthly.dropna(how="all")
df_monthly.tail()


## 4. Overview: Search Interest vs. Price

Monthly long history first (regime-level view, GME's squeeze should be unmistakable), then the
recent daily window (higher resolution, no long-run trend to distract from short-term co-movement).


In [ ]:
fig, axes = plt.subplots(len(COMPANIES), 1, figsize=(13, 4 * len(COMPANIES)), sharex=True)
for ax, t in zip(axes, COMPANIES):
    price = monthly_close[f"{t}_close"]
    ax.plot(price.index, price / price.iloc[0] * 100, label=f"{t} price (indexed)", color="tab:blue")
    ax2 = ax.twinx()
    ax2.plot(trends_monthly[t].index, trends_monthly[t]["interest"], label=f"{t} search interest",
              color="tab:orange", alpha=0.7)
    ax.set_title(f"{t}: Monthly Price vs. Google Trends Search Interest (2019-present)")
    l1, lb1 = ax.get_legend_handles_labels()
    l2, lb2 = ax2.get_legend_handles_labels()
    ax.legend(l1 + l2, lb1 + lb2, loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(len(COMPANIES), 1, figsize=(13, 4 * len(COMPANIES)), sharex=True)
for ax, t in zip(axes, COMPANIES):
    price = df_daily[f"{t}_close"].dropna()
    ax.plot(price.index, price / price.iloc[0] * 100, label=f"{t} price (indexed)", color="tab:blue")
    ax2 = ax.twinx()
    ax2.plot(df_daily.index, df_daily[f"{t}_trends"], label=f"{t} search interest",
              color="tab:orange", alpha=0.7)
    ax.set_title(f"{t}: Daily Price vs. Google Trends Search Interest (last ~9 months)")
    l1, lb1 = ax.get_legend_handles_labels()
    l2, lb2 = ax2.get_legend_handles_labels()
    ax.legend(l1 + l2, lb1 + lb2, loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()


## 5. Returns & Correlation

As with the Wikipedia notebook, we use search-interest **growth** (log change), not the raw level,
since search volume trends upward over time as a company grows regardless of any trading signal.


In [ ]:
daily_rets = pd.DataFrame(index=df_daily.index)
for t in COMPANIES:
    daily_rets[f"{t}_ret"] = np.log(df_daily[f"{t}_close"]).diff()
    daily_rets[f"{t}_trends_chg"] = np.log(df_daily[f"{t}_trends"].replace(0, np.nan)).diff()
daily_rets = daily_rets.dropna(how="all")

for t in COMPANIES:
    df_monthly[f"{t}_trends_chg"] = np.log(df_monthly[f"{t}_trends"].replace(0, np.nan)).diff()

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, t in zip(axes.flat, COMPANIES):
    sub = daily_rets[[f"{t}_ret", f"{t}_trends_chg"]].dropna()
    corr_val = sub.corr().iloc[0, 1]
    sns.heatmap(sub.corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax, cbar=False)
    ax.set_title(f"{t} (daily): return vs. trends-change corr = {corr_val:.3f}")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, t in zip(axes.flat, COMPANIES):
    sub = df_monthly[[f"{t}_ret", f"{t}_trends_chg"]].dropna()
    corr_val = sub.corr().iloc[0, 1] if len(sub) > 2 else np.nan
    sns.heatmap(sub.corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax, cbar=False)
    ax.set_title(f"{t} (monthly, 2019+): return vs. trends-change corr = {corr_val:.3f}")
plt.tight_layout()
plt.show()


## 6. Lead-Lag Analysis & Granger Causality (Daily Window)

Does search interest lead price (research before buying) or does price lead search interest (search
because of news)? Uses the daily window, since a 1-2 day lag is meaningless at monthly resolution.


In [ ]:
def lagged_corr(y, x, max_lag=5):
    return pd.Series({lag: y.corr(x.shift(lag)) for lag in range(-max_lag, max_lag + 1)})

max_lag = 5
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, t in zip(axes.flat, COMPANIES):
    sub = daily_rets[[f"{t}_ret", f"{t}_trends_chg"]].dropna()
    ll = lagged_corr(sub[f"{t}_ret"], sub[f"{t}_trends_chg"], max_lag)
    colors = ["#d62728" if l == 0 else "#1f77b4" for l in ll.index]
    ll.plot(kind="bar", ax=ax, color=colors)
    ax.set_title(f"{t}: trends-change (lag) vs return (t)")
    ax.set_xlabel("Lag (positive = search interest leads price)")
plt.tight_layout()
plt.show()


In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

for t in COMPANIES:
    print(f"\n=== {t}: does search-interest change Granger-cause return? ===")
    sub = daily_rets[[f"{t}_ret", f"{t}_trends_chg"]].dropna()
    try:
        grangercausalitytests(sub, maxlag=3, verbose=True)
    except Exception as e:
        print(f"  skipped ({e})")


## 7. Cross-Validation: Google Trends vs. Wikipedia Pageviews

The real test: two **independently collected** attention proxies for the same companies. We refetch
Wikipedia pageviews here (same free Wikimedia API used in the companion notebook) over matching
windows and directly correlate the two proxies' growth rates against each other, per ticker. High
agreement is much stronger evidence of a real "attention" signal than either dataset alone; low
agreement suggests at least one of them is mostly noise for this purpose.


In [ ]:
WIKI_ARTICLES = {"TSLA": "Tesla,_Inc.", "GME": "GameStop", "COIN": "Coinbase", "AAPL": "Apple_Inc."}
WIKI_USER_AGENT = "googleTrendsResearch-notebook/1.0 (personal research; contact: bruno.inzunza24@gmail.com)"

# Wikimedia's pageviews API has occasional article-specific data gaps at certain start dates
# (verified: GameStop's article 404s for a 2019-01-01 start specifically, but is fine from
# 2020-01-01 — likely an artifact of a page-move in its edit history). 2020-01-01 is the same
# start already verified working for all four articles in the companion notebook, so we use it
# here too rather than chase Wikimedia's exact quirk; it costs one year of overlap with the
# Trends monthly history (2019+) but keeps this section reliable.
WIKI_START = "2020-01-01"


def fetch_wiki_pageviews(article, start, end, project="en.wikipedia"):
    url = (
        f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
        f"{project}/all-access/user/{requests.utils.quote(article, safe='')}/daily/{start}00/{end}00"
    )
    r = requests.get(url, headers={"User-Agent": WIKI_USER_AGENT}, timeout=20)
    r.raise_for_status()
    items = r.json()["items"]
    d = pd.DataFrame(items)[["timestamp", "views"]]
    d["date"] = pd.to_datetime(d["timestamp"], format="%Y%m%d%H").dt.normalize()
    return d.set_index("date")["views"]


pageviews_monthly, pageviews_daily = {}, {}
for t, article in WIKI_ARTICLES.items():
    pv = fetch_wiki_pageviews(article, WIKI_START.replace("-", ""), TODAY.strftime("%Y%m%d"))
    pageviews_monthly[t] = pv.resample("MS").sum()
    pageviews_daily[t] = pv.reindex(pd.date_range(DAILY_START, TODAY, freq="D")).fillna(0)
    time.sleep(0.3)


In [ ]:
cross_val_rows = []
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, t in zip(axes.flat, COMPANIES):
    trends_g = np.log(trends_monthly[t]["interest"].replace(0, np.nan)).diff()
    wiki_g = np.log(pageviews_monthly[t].replace(0, np.nan)).diff()
    sub = pd.DataFrame({"trends_chg": trends_g, "wiki_chg": wiki_g}).dropna()
    corr_val = sub["trends_chg"].corr(sub["wiki_chg"]) if len(sub) > 2 else np.nan
    ax.scatter(sub["trends_chg"], sub["wiki_chg"], alpha=0.5, s=15)
    ax.set_title(f"{t} (monthly): Trends vs. Wikipedia growth, corr={corr_val:.3f}")
    ax.set_xlabel("Google Trends WoW change")
    ax.set_ylabel("Wikipedia pageviews WoW change")
    cross_val_rows.append({"ticker": t, "granularity": "monthly", "corr": corr_val, "n": len(sub)})
plt.tight_layout()
plt.show()

pd.DataFrame(cross_val_rows)


In [ ]:
cross_val_daily_rows = []
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, t in zip(axes.flat, COMPANIES):
    trends_g = daily_rets[f"{t}_trends_chg"]
    wiki_daily_rolled = roll_forward_to_trading_days(pageviews_daily[t], trading_days).reindex(trading_days)
    wiki_g = np.log(wiki_daily_rolled.replace(0, np.nan)).diff()
    sub = pd.DataFrame({"trends_chg": trends_g, "wiki_chg": wiki_g}).dropna()
    corr_val = sub["trends_chg"].corr(sub["wiki_chg"]) if len(sub) > 2 else np.nan
    ax.scatter(sub["trends_chg"], sub["wiki_chg"], alpha=0.4, s=12)
    ax.set_title(f"{t} (daily): Trends vs. Wikipedia growth, corr={corr_val:.3f}")
    ax.set_xlabel("Google Trends daily change")
    ax.set_ylabel("Wikipedia pageviews daily change")
    cross_val_daily_rows.append({"ticker": t, "granularity": "daily", "corr": corr_val, "n": len(sub)})
plt.tight_layout()
plt.show()

pd.DataFrame(cross_val_daily_rows)


## 8. Event Study: GME's Squeeze, Seen Through Search Interest

The companion notebook found this via Wikipedia pageview shocks; here we look for the same episode in
monthly Google Trends data as an independent confirmation that both proxies capture the same real
event, not idiosyncratic noise in one dataset.


In [ ]:
fig, ax = plt.subplots()
gme_price = monthly_close["GME_close"]
ax.plot(gme_price.index, gme_price, color="tab:blue", label="GME monthly close")
ax2 = ax.twinx()
ax2.plot(trends_monthly["GME"].index, trends_monthly["GME"]["interest"], color="tab:orange",
          label="GME search interest")
ax.set_title("GME: Monthly Price vs. Google Trends Search Interest, 2019-present")
l1, lb1 = ax.get_legend_handles_labels()
l2, lb2 = ax2.get_legend_handles_labels()
ax.legend(l1 + l2, lb1 + lb2, loc="upper left")
plt.tight_layout()
plt.show()

peak_month = trends_monthly["GME"]["interest"].idxmax()
print(f"Peak GME search-interest month: {peak_month.date()} (value={trends_monthly['GME']['interest'].max()})")


## 9. Toy Backtest: Trade the Search-Interest Shock (Daily Window)

Same construct as the Wikipedia notebook's event study/backtest: flag days where search-interest
growth's z-score is extreme, then check whether momentum or reversal followed. No transaction costs
modeled — a raw-signal check on the short daily window only (the long monthly history has too few
independent monthly observations for a meaningful backtest).


In [ ]:
def zscore(s, window=30):
    return (s - s.rolling(window).mean()) / s.rolling(window).std()


def sharpe(returns, periods_per_year=252):
    r = returns.dropna()
    if len(r) == 0 or r.std() == 0:
        return np.nan
    return (r.mean() / r.std()) * np.sqrt(periods_per_year)


shock_threshold = 2.0
summary_rows = []
for t in COMPANIES:
    z = zscore(daily_rets[f"{t}_trends_chg"])
    shock = (z > shock_threshold).astype(float)

    momentum_ret = shock.shift(1) * daily_rets[f"{t}_ret"]
    reversal_ret = -shock.shift(1) * daily_rets[f"{t}_ret"]
    buy_hold = daily_rets[f"{t}_ret"]

    summary_rows.append({
        "ticker": t,
        "n_shock_days": int(shock.sum()),
        "momentum_sharpe": sharpe(momentum_ret),
        "reversal_sharpe": sharpe(reversal_ret),
        "buy_hold_sharpe": sharpe(buy_hold),
    })

pd.DataFrame(summary_rows).set_index("ticker")


## 10. Takeaways & Next Steps

- **Read Section 7 first**: if Google Trends and Wikipedia pageviews *don't* agree with each other
  for a given ticker, be skeptical of whatever either one shows in isolation for that ticker — the
  disagreement itself is the finding. If they *do* agree, that's a meaningfully stronger basis for
  believing there's a real attention effect than either notebook alone provided.
- **The daily window is short by construction** (~9 months, capped by Google's own resolution limit)
  — re-run Section 9's backtest periodically as more data accumulates rather than trusting a single
  9-month sample.
- **`pytrends` is unofficial**: if this notebook starts failing outright (not just rate-limiting),
  check whether Google changed something server-side — search "pytrends broken" for the current
  workaround, or consider a paid alternative (SerpApi, DataForSEO) for production use.
- **Entity disambiguation matters**: if you add tickers, look up and verify each one's `mid` code via
  Section 1's `suggestions()` output rather than assuming a raw company-name search term is
  unambiguous — the Apple/apple collision alone would have silently corrupted results here.
- **Robustness**: as with every notebook in this repo, validate out-of-sample and account for
  transaction costs before treating anything here as an actual edge.
